# How concentrated liabilities amplify systemic loss

## Comparing row concentration through majorization

For a debtor $i$, let $\pi_i^{\downarrow}$ be its relative-liability row sorted from largest to smallest. The concentrated row majorizes the dispersed row when

$$
\sum_{j=1}^{k}\pi_{i,j}^{\mathrm{concentrated},\downarrow}
\geq
\sum_{j=1}^{k}\pi_{i,j}^{\mathrm{dispersed},\downarrow},
\qquad k=1,\ldots,n-1,
$$

and both rows have the same total. Here, $k$ counts the largest creditor shares. The ring places all internal liability on one creditor, while the complete network spreads the same internal total over eleven creditors. This ordering describes the rows; it does not determine systemic loss without further conditions on assets, liabilities, and their ordering.

In [1]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)
from scipy.optimize import linprog


BOOK_ROOT = Path.cwd().parents[1]

import RiskLabAI
from RiskLabAI.optimization import eisenberg_noe_clearing

assert RiskLabAI.__version__ == "3.2.0"

FIGURES = BOOK_ROOT / "outputs" / "figures" / "28_network_clearing"
TABLES = BOOK_ROOT / "outputs" / "tables" / "28_network_clearing"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "28_network_clearing"
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def ring_matrix(size, internal_share=0.8):
    matrix = np.zeros((size, size))
    for debtor in range(size):
        matrix[debtor, (debtor + 1) % size] = internal_share
    return matrix


def complete_matrix(size, internal_share=0.8):
    matrix = np.full((size, size), internal_share / (size - 1))
    np.fill_diagonal(matrix, 0)
    return matrix


def core_periphery_matrix(size, internal_share=0.8, core_size=4):
    matrix = np.zeros((size, size))
    core = np.arange(core_size)
    periphery = np.arange(core_size, size)
    for debtor in range(size):
        if debtor < core_size:
            other_core = core[core != debtor]
            matrix[debtor, other_core] = 0.4 / len(other_core)
            matrix[debtor, periphery] = 0.4 / len(periphery)
        else:
            matrix[debtor, core] = 0.64 / len(core)
            other_periphery = periphery[periphery != debtor]
            matrix[debtor, other_periphery] = 0.16 / len(other_periphery)
    return matrix * (internal_share / 0.8)


def community_matrix(size, internal_share=0.8):
    matrix = np.zeros((size, size))
    half = size // 2
    for debtor in range(size):
        own = np.arange(0, half) if debtor < half else np.arange(half, size)
        other = np.arange(half, size) if debtor < half else np.arange(0, half)
        own = own[own != debtor]
        matrix[debtor, own] = 0.68 / len(own)
        matrix[debtor, other] = 0.12 / len(other)
    return matrix * (internal_share / 0.8)


def greatest_iteration(external_assets, nominal_liabilities, relative_liabilities, tolerance=1e-12, maximum_steps=10000):
    payments = np.asarray(nominal_liabilities, dtype=float).copy()
    history = [payments.copy()]
    for _ in range(maximum_steps):
        updated = np.minimum(nominal_liabilities, external_assets + relative_liabilities.T @ payments)
        history.append(updated.copy())
        if np.max(np.abs(updated - payments)) <= tolerance:
            return updated, np.asarray(history)
        payments = updated
    raise RuntimeError("Monotone iteration did not converge.")


def direct_linear_program(external_assets, nominal_liabilities, relative_liabilities):
    result = linprog(
        -np.ones(len(nominal_liabilities)),
        A_ub=np.eye(len(nominal_liabilities)) - relative_liabilities.T,
        b_ub=np.minimum(external_assets, nominal_liabilities),
        bounds=np.column_stack((np.zeros(len(nominal_liabilities)), nominal_liabilities)),
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)
    return np.asarray(result.x)


def clearing_metrics(external_assets, nominal_liabilities, relative_liabilities):
    payments = eisenberg_noe_clearing(external_assets, nominal_liabilities, relative_liabilities)
    fixed = np.minimum(nominal_liabilities, external_assets + relative_liabilities.T @ payments)
    unpaid = np.maximum(nominal_liabilities - payments, 0)
    return {
        "payments": payments,
        "unpaid": unpaid,
        "total payments": float(payments.sum()),
        "total unpaid": float(unpaid.sum()),
        "default count": int(np.sum(unpaid > 1e-8)),
        "payment ratio": float(payments.sum() / nominal_liabilities.sum()) if nominal_liabilities.sum() else 1.0,
        "fixed point residual": float(np.max(np.abs(payments - fixed))) if len(payments) else 0.0,
    }


def network_positions(name, size):
    if name in {"Ring", "Complete"}:
        return nx.circular_layout(range(size))
    if name == "Core-periphery":
        positions = {}
        core = nx.circular_layout(range(4), scale=0.45)
        outer = nx.circular_layout(range(4, size), scale=1.0)
        positions.update(core)
        positions.update(outer)
        return positions
    positions = {}
    left = nx.circular_layout(range(size // 2), scale=0.55, center=(-0.7, 0))
    right = nx.circular_layout(range(size // 2, size), scale=0.55, center=(0.7, 0))
    positions.update(left)
    positions.update(right)
    return positions


def plot_network(relative_liabilities, name, stem, node_values=None):
    graph = nx.DiGraph()
    size = len(relative_liabilities)
    graph.add_nodes_from(range(size))
    for debtor in range(size):
        for creditor in range(size):
            weight = relative_liabilities[debtor, creditor]
            if weight > 1e-12:
                graph.add_edge(debtor, creditor, weight=weight)
    positions = network_positions(name, size)
    values = np.ones(size) if node_values is None else np.asarray(node_values)
    colors = [PALETTE["rose"] if value > 1e-8 else PALETTE["cyan"] for value in values]
    widths = [0.8 + 5 * graph.edges[edge]["weight"] for edge in graph.edges]
    figure, axis = plt.subplots(figsize=(7.2, 6.0), constrained_layout=True)
    nx.draw_networkx_nodes(graph, positions, node_color=colors, node_size=620, edgecolors=PALETTE["text"], linewidths=0.8, ax=axis)
    nx.draw_networkx_labels(graph, positions, font_color=PALETTE["background"], font_weight="bold", ax=axis)
    nx.draw_networkx_edges(graph, positions, width=widths, alpha=0.55, edge_color=PALETTE["text"], arrows=True, arrowsize=13, connectionstyle="arc3,rad=0.06", ax=axis)
    axis.set_title(name)
    axis.set_axis_off()
    save_figure(figure, stem)
    edge_data = pd.DataFrame([{"Debtor": source, "Creditor": target, "Relative liability": graph.edges[source, target]["weight"]} for source, target in graph.edges])
    edge_data.to_csv(FIGURE_DATA / f"{stem}.csv", index=False)


def majorizes(source, target, tolerance=1e-12):
    source_sorted = np.sort(np.asarray(source))[::-1]
    target_sorted = np.sort(np.asarray(target))[::-1]
    return bool(np.all(np.cumsum(source_sorted)[:-1] >= np.cumsum(target_sorted)[:-1] - tolerance) and np.isclose(source_sorted.sum(), target_sorted.sum(), atol=tolerance))

size = 12
internal_share = 0.8
concentrated = ring_matrix(size, internal_share)
dispersed = complete_matrix(size, internal_share)

summary = pd.DataFrame([
    {
        "Structure": "Concentrated ring",
        "Internal row total": concentrated.sum(axis=1).mean(),
        "Positive creditors per row": int(np.sum(concentrated[0] > 0)),
        "Mean squared row concentration": np.mean(np.sum(concentrated ** 2, axis=1)),
        "Largest creditor share": concentrated.max(),
    },
    {
        "Structure": "Dispersed complete",
        "Internal row total": dispersed.sum(axis=1).mean(),
        "Positive creditors per row": int(np.sum(dispersed[0] > 0)),
        "Mean squared row concentration": np.mean(np.sum(dispersed ** 2, axis=1)),
        "Largest creditor share": dispersed.max(),
    },
])
save_table(summary, "concentration_summary", "Row totals and concentration measures for the two liability structures.", "tab:chapter29-concentration-summary", fit=True)

majorization_rows = []
for debtor in range(size):
    source = np.sort(concentrated[debtor])[::-1]
    target = np.sort(dispersed[debtor])[::-1]
    gaps = np.cumsum(source) - np.cumsum(target)
    majorization_rows.append({
        "Debtor": debtor,
        "Smallest proper cumulative gap": max(float(gaps[:-1].min()), 0.0),
        "Total difference": 0.0 if abs(float(gaps[-1])) <= 1e-12 else float(gaps[-1]),
        "Concentrated majorizes dispersed": majorizes(source, target),
    })
save_table(pd.DataFrame(majorization_rows), "row_majorization_checks", "Majorization checks for every generated liability row.", "tab:chapter29-row-majorization", fit=True)

conditions = pd.DataFrame([
    {"Statement": "The concentrated row majorizes the dispersed row", "Established here": "Yes", "Limit": "Ordinary row ordering only"},
    {"Statement": "Concentration always raises systemic loss", "Established here": "No", "Limit": "Published conditions are required"},
    {"Statement": "Concentration always lowers systemic loss", "Established here": "No", "Limit": "Published conditions are required"},
    {"Statement": "The numerical comparison estimates a real banking system", "Established here": "No", "Limit": "Generated balance sheets only"},
])
save_table(conditions, "concentration_conditions", "Claims established and not established by row majorization.", "tab:chapter29-concentration-conditions", fit=True)

cumulative = pd.DataFrame({
    "Rank": np.arange(1, size + 1),
    "Concentrated cumulative share": np.cumsum(np.sort(concentrated[0])[::-1]),
    "Dispersed cumulative share": np.cumsum(np.sort(dispersed[0])[::-1]),
})
cumulative.to_csv(FIGURE_DATA / "row_cumulative_concentration.csv", index=False)
figure, axis = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
axis.step(cumulative["Rank"], cumulative["Concentrated cumulative share"], where="post", linewidth=2.1, color=PALETTE["rose"], label="Concentrated ring")
axis.step(cumulative["Rank"], cumulative["Dispersed cumulative share"], where="post", linewidth=2.1, color=PALETTE["cyan"], label="Dispersed complete")
axis.set_xlabel("Number of largest creditor shares")
axis.set_ylabel("Cumulative internal liability share")
axis.set_title("The Ring Row Majorizes the Complete-Network Row")
axis.legend()
finish_axis(axis)
save_figure(figure, "row_cumulative_concentration")

,Structure,Internal row total,Positive creditors per row,Mean squared row concentration,Largest creditor share
0,Concentrated ring,0.8,1,0.64,0.8
1,Dispersed complete,0.8,11,0.058182,0.072727


,Debtor,Smallest proper cumulative gap,Total difference,Concentrated majorizes dispersed
0,0,0,0,Pass
1,1,0,0,Pass
2,2,0,0,Pass
3,3,0,0,Pass
4,4,0,0,Pass
5,5,0,0,Pass
6,6,0,0,Pass
7,7,0,0,Pass
8,8,0,0,Pass
9,9,0,0,Pass


,Statement,Established here,Limit
0,The concentrated row majorizes the dispersed row,Yes,Ordinary row ordering only
1,Concentration always raises systemic loss,No,Published conditions are required
2,Concentration always lowers systemic loss,No,Published conditions are required
3,The numerical comparison estimates a real bank...,No,Generated balance sheets only


<Figure size 880x520 with 1 Axes>

## Two Hundred Generated Comparisons

For generated balance sheet $b$, define the loss difference

$$
\Delta_b
=
\sum_{i=1}^{12}\left(\overline p_{b,i}-p^{\mathrm{concentrated}}_{b,i}\right)
-
\sum_{i=1}^{12}\left(\overline p_{b,i}-p^{\mathrm{dispersed}}_{b,i}\right).
$$

The value $\Delta_b$ is positive when the concentrated ring has more unpaid liability and negative when it has less. Each run changes external assets and liability sizes but preserves the same $0.8$ internal row total. The experiment looks for both directions rather than imposing a winner.

In [2]:
comparison_rows = []
saved_inputs = {}
for draw in range(200):
    rng = np.random.default_rng(290300 + draw)
    liabilities = 100 * rng.lognormal(mean=0.0, sigma=0.75, size=size)
    liabilities = liabilities / liabilities.mean()
    asset_ratio = rng.uniform(0.02, 0.75, size)
    external_assets = liabilities * asset_ratio
    concentrated_result = clearing_metrics(external_assets, liabilities, concentrated)
    dispersed_result = clearing_metrics(external_assets, liabilities, dispersed)
    difference = concentrated_result["total unpaid"] - dispersed_result["total unpaid"]
    rank_correlation = pd.Series(liabilities).corr(pd.Series(external_assets), method="spearman")
    comparison_rows.append({
        "Draw": draw,
        "Concentrated unpaid": concentrated_result["total unpaid"],
        "Dispersed unpaid": dispersed_result["total unpaid"],
        "Concentrated minus dispersed": difference,
        "Concentrated defaults": concentrated_result["default count"],
        "Dispersed defaults": dispersed_result["default count"],
        "Liability coefficient of variation": liabilities.std(ddof=1) / liabilities.mean(),
        "Asset-liability rank correlation": rank_correlation,
        "Maximum fixed point residual": max(concentrated_result["fixed point residual"], dispersed_result["fixed point residual"]),
    })
    saved_inputs[draw] = (external_assets, liabilities, concentrated_result, dispersed_result)

comparisons = pd.DataFrame(comparison_rows)
assert (comparisons["Concentrated minus dispersed"] > 1e-8).any()
assert (comparisons["Concentrated minus dispersed"] < -1e-8).any()

seed_contract = pd.DataFrame([
    {"Setting": "Generated balance sheets", "Value": 200},
    {"Setting": "First seed", "Value": 290300},
    {"Setting": "Last seed", "Value": 290499},
    {"Setting": "Institutions", "Value": size},
    {"Setting": "Internal row total", "Value": internal_share},
    {"Setting": "Liability lognormal scale", "Value": 0.75},
    {"Setting": "External asset ratio range", "Value": "0.02 to 0.75"},
])
save_table(seed_contract, "concentration_seed_contract", "Fixed generation settings for liability-concentration comparisons.", "tab:chapter29-concentration-seeds")

positive = int(np.sum(comparisons["Concentrated minus dispersed"] > 1e-8))
negative = int(np.sum(comparisons["Concentrated minus dispersed"] < -1e-8))
equal = int(len(comparisons) - positive - negative)
directions = pd.DataFrame([
    {"Loss direction": "Concentrated higher", "Generated runs": positive, "Share": positive / len(comparisons)},
    {"Loss direction": "Concentrated lower", "Generated runs": negative, "Share": negative / len(comparisons)},
    {"Loss direction": "Numerically equal", "Generated runs": equal, "Share": equal / len(comparisons)},
])
save_table(directions, "concentration_direction_counts", "Observed loss directions across the generated comparisons.", "tab:chapter29-concentration-directions")

lower_draw = int(comparisons["Concentrated minus dispersed"].idxmin())
higher_draw = int(comparisons["Concentrated minus dispersed"].idxmax())
extremes = comparisons.loc[[lower_draw, higher_draw], ["Concentrated unpaid", "Dispersed unpaid", "Concentrated minus dispersed", "Concentrated defaults", "Dispersed defaults"]].copy()
extremes.insert(1, "Case", ["Concentrated lower", "Concentrated higher"])
save_table(extremes, "concentration_extreme_cases", "Generated cases showing opposite finite loss directions.", "tab:chapter29-concentration-extremes", fit=True)

node_rows = []
for draw, case in [(lower_draw, "Concentrated lower"), (higher_draw, "Concentrated higher")]:
    external_assets, liabilities, concentrated_result, dispersed_result = saved_inputs[draw]
    for institution in range(size):
        node_rows.append({
            "Case": case,
            "Draw": draw,
            "Institution": institution,
            "External assets": external_assets[institution],
            "Nominal liabilities": liabilities[institution],
            "Concentrated unpaid": concentrated_result["unpaid"][institution],
            "Dispersed unpaid": dispersed_result["unpaid"][institution],
        })
save_table(pd.DataFrame(node_rows), "concentration_selected_nodes", "Institution-level inputs and unpaid amounts for the two selected generated cases.", "tab:chapter29-concentration-selected-nodes", fit=True)

distribution = pd.DataFrame([
    {"Statistic": "Mean", "Value": comparisons["Concentrated minus dispersed"].mean()},
    {"Statistic": "Standard deviation", "Value": comparisons["Concentrated minus dispersed"].std(ddof=1)},
    {"Statistic": "Minimum", "Value": comparisons["Concentrated minus dispersed"].min()},
    {"Statistic": "25th percentile", "Value": comparisons["Concentrated minus dispersed"].quantile(0.25)},
    {"Statistic": "Median", "Value": comparisons["Concentrated minus dispersed"].median()},
    {"Statistic": "75th percentile", "Value": comparisons["Concentrated minus dispersed"].quantile(0.75)},
    {"Statistic": "Maximum", "Value": comparisons["Concentrated minus dispersed"].max()},
])
save_table(distribution, "concentration_distribution_summary", "Distribution of concentrated-minus-dispersed unpaid liability.", "tab:chapter29-concentration-distribution")

comparisons.to_csv(FIGURE_DATA / "concentration_loss_differences.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
axis.hist(comparisons["Concentrated minus dispersed"], bins=24, color=PALETTE["violet"], edgecolor=PALETTE["text"], alpha=0.88)
axis.axvline(0, color=PALETTE["amber"], linewidth=1.8)
axis.set_xlabel("Concentrated minus dispersed unpaid liability")
axis.set_ylabel("Generated balance sheets")
axis.set_title("The Finite Loss Direction Changes across Generated Inputs")
finish_axis(axis)
save_figure(figure, "concentration_loss_differences")

scatter = comparisons[["Asset-liability rank correlation", "Liability coefficient of variation", "Concentrated minus dispersed"]].copy()
scatter.to_csv(FIGURE_DATA / "concentration_ordering_scatter.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
points = axis.scatter(scatter["Asset-liability rank correlation"], scatter["Concentrated minus dispersed"], c=scatter["Liability coefficient of variation"], cmap="viridis", s=34, alpha=0.85)
axis.axhline(0, color=PALETTE["amber"], linewidth=1.5)
axis.set_xlabel("Asset-liability rank correlation")
axis.set_ylabel("Concentrated minus dispersed unpaid liability")
axis.set_title("Loss Differences Depend on More Than Row Concentration")
colorbar = figure.colorbar(points, ax=axis)
colorbar.set_label("Liability coefficient of variation")
finish_axis(axis)
save_figure(figure, "concentration_ordering_scatter")

bar_rows = []
for draw, case in [(lower_draw, "Concentrated lower"), (higher_draw, "Concentrated higher")]:
    row = comparisons.loc[draw]
    bar_rows.extend([
        {"Case": case, "Structure": "Concentrated ring", "Total unpaid": row["Concentrated unpaid"]},
        {"Case": case, "Structure": "Dispersed complete", "Total unpaid": row["Dispersed unpaid"]},
    ])
bar_data = pd.DataFrame(bar_rows)
bar_data.to_csv(FIGURE_DATA / "concentration_reversal_examples.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.4, 4.7), constrained_layout=True)
for axis, case in zip(axes, ["Concentrated lower", "Concentrated higher"]):
    part = bar_data.loc[bar_data["Case"] == case]
    axis.bar(part["Structure"], part["Total unpaid"], color=[PALETTE["rose"], PALETTE["cyan"]])
    axis.set_title(case)
    axis.set_ylabel("Total unpaid liabilities")
    axis.tick_params(axis="x", rotation=16)
    finish_axis(axis)
save_figure(figure, "concentration_reversal_examples")

,Setting,Value
0,Generated balance sheets,200
1,First seed,"290,300"
2,Last seed,"290,499"
3,Institutions,12
4,Internal row total,0.8
5,Liability lognormal scale,0.75
6,External asset ratio range,0.02 to 0.75


,Loss direction,Generated runs,Share
0,Concentrated higher,182,0.91
1,Concentrated lower,18,0.09
2,Numerically equal,0,0


,Concentrated unpaid,Case,Dispersed unpaid,Concentrated minus dispersed,Concentrated defaults,Dispersed defaults
100,1.531058,Concentrated lower,2.780181,-1.249123,3,5
154,2.835146,Concentrated higher,0.729894,2.105252,6,4


,Case,Draw,Institution,External assets,Nominal liabilities,Concentrated unpaid,Dispersed unpaid
0,Concentrated lower,100,0,0.204643,0.68518,0,0
1,Concentrated lower,100,1,0.512875,0.9655,0,0
2,Concentrated lower,100,2,0.862521,2.552502,0.917581,1.123384
3,Concentrated lower,100,3,0.350049,1.555114,2.22045e-16,0.603725
4,Concentrated lower,100,4,0.075773,1.280066,0,0.584357
5,Concentrated lower,100,5,0.332071,1.213316,0,0.278686
6,Concentrated lower,100,6,0.373889,0.569576,0,0
7,Concentrated lower,100,7,0.27214,0.621311,0,0
8,Concentrated lower,100,8,0.226963,0.334852,0,0
9,Concentrated lower,100,9,0.111491,0.919033,0.539661,0.190028


,Statistic,Value
0,Mean,0.679932
1,Standard deviation,0.572194
2,Minimum,-1.249123
3,25th percentile,0.362382
4,Median,0.671372
5,75th percentile,1.039247
6,Maximum,2.105252


<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 2 Axes>

<Figure size 1040x470 with 2 Axes>

## Recalculating outcomes within model scope

For every saved comparison, the fixed point residual is

$$
r_b=\max_i\left|p_{b,i}-\min\!\left(\overline p_{b,i},e_{b,i}+(\Pi^{\mathsf T}p_b)_i\right)\right|.
$$

The residual $r_b$ checks the numerical clearing identity. Recomputing selected cases with monotone iteration checks the released result by a separate route. Neither check turns a mathematical link into evidence that one institution caused another institution's observed loss.

In [3]:
recompute_rows = []
for draw, case in [(lower_draw, "Concentrated lower"), (higher_draw, "Concentrated higher")]:
    external_assets, liabilities, concentrated_result, dispersed_result = saved_inputs[draw]
    for name, matrix, result in [
        ("Concentrated ring", concentrated, concentrated_result),
        ("Dispersed complete", dispersed, dispersed_result),
    ]:
        iterated, history = greatest_iteration(external_assets, liabilities, matrix)
        recompute_rows.append({
            "Case": case,
            "Structure": name,
            "Iteration steps": len(history) - 1,
            "Released versus iteration residual": np.max(np.abs(result["payments"] - iterated)),
            "Fixed point residual": result["fixed point residual"],
            "Pass": bool(np.allclose(result["payments"], iterated, atol=1e-8)),
        })
save_table(pd.DataFrame(recompute_rows), "recomputation_checks", "Independent recalculation checks for the two selected generated cases.", "tab:chapter29-recomputation-checks", fit=True)

scope = pd.DataFrame([
    {"Question": "Do concentrated rows majorize dispersed rows?", "Answered": "Yes", "Boundary": "The supplied ring and complete rows"},
    {"Question": "Is one loss direction universal?", "Answered": "No", "Boundary": "Both directions occur in generated cases"},
    {"Question": "Are published balancing conditions tested?", "Answered": "No", "Boundary": "No theorem extension"},
    {"Question": "Do links prove causal spillovers?", "Answered": "No", "Boundary": "No causal design or intervention"},
    {"Question": "Are these actual banking-system estimates?", "Answered": "No", "Boundary": "Generated data only"},
])
save_table(scope, "causal_scope_boundaries", "Interpretation boundaries for the liability-concentration experiment.", "tab:chapter29-causal-boundaries", fit=True)

,Case,Structure,Iteration steps,Released versus iteration residual,Fixed point residual,Pass
0,Concentrated lower,Concentrated ring,3,2.22045e-16,2.22045e-16,Pass
1,Concentrated lower,Dispersed complete,23,2.58127e-13,1.11022e-16,Pass
2,Concentrated higher,Concentrated ring,4,1.11022e-16,1.11022e-16,Pass
3,Concentrated higher,Dispersed complete,18,2.29261e-13,4.44089e-16,Pass


,Question,Answered,Boundary
0,Do concentrated rows majorize dispersed rows?,Yes,The supplied ring and complete rows
1,Is one loss direction universal?,No,Both directions occur in generated cases
2,Are published balancing conditions tested?,No,No theorem extension
3,Do links prove causal spillovers?,No,No causal design or intervention
4,Are these actual banking-system estimates?,No,Generated data only


,Question,Answered,Boundary
0,Do concentrated rows majorize dispersed rows?,Yes,The supplied ring and complete rows
1,Is one loss direction universal?,No,Both directions occur in generated cases
2,Are published balancing conditions tested?,No,No theorem extension
3,Do links prove causal spillovers?,No,No causal design or intervention
4,Are these actual banking-system estimates?,No,Generated data only
